# NB11 — Q3 — Does compute-need transfer between architectures?

        **CPU only — turn the accelerator OFF. ~15 minutes. One account.**


> **New here?** Read `05_PLAIN_ENGLISH_GUIDE.md` first — it explains what this
> project is measuring and why, without jargon. This notebook assumes you have.


        ## The question

        > Does a ResNet agree with a Vision Transformer about which images need more computation?

        ## In plain English

        **This is the main question of the project.** We take every pair of architectures and measure how much they agree — then divide by the noise ceiling from NB09, so the number means 'how much of the achievable agreement did we actually get' rather than a raw correlation of unknown scale.

        ## Why it matters

        If compute-need transfers, a big teacher model can tell a small student how much effort each image deserves — and that's a useful method. If it doesn't transfer, a growing line of teacher-guided adaptive-inference work rests on a false premise, and demonstrating that clearly is the stronger paper.

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   ee2ce7f3d2a6   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  6abdba4ff104   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

## Step 1 — Load

In [ ]:
ACCOUNT = 'acct1'      # <<< CHANGE ME

sess = msc.Session(account=ACCOUNT, phase='analysis', dataset='cifar100',
                   enable_hf=True)
# Metrics and per-sample tables only -- checkpoints excluded. Fast.
sess.sync_state(include_checkpoints=False, verbose=True)

import pandas as pd, numpy as np, matplotlib.pyplot as plt

# Inventory: what do we actually have to work with?
runs = {}
for d in sorted(sess.runs_dir.iterdir()) if sess.runs_dir.exists() else []:
    ps = d / 'per_sample'
    m = msc.read_json(ps / 'meta.json', default=None)
    if m and (ps / 'test.parquet').exists():
        runs[d.name] = m
budgets = {r: sess.budgets(m['arch']) for r, m in runs.items()}

inv = pd.DataFrame([{'run_id': k, 'arch': v['arch'], 'family': v['family'],
                     'seed': v['seed'],
                     'order': v['sample_order_hash'][:10]} for k, v in runs.items()])
if len(inv):
    inv = inv.sort_values(['family', 'arch', 'seed'])
    display(inv)
    print(f"\n{len(runs)} measured models   "
          f"{inv.order.nunique()} distinct image orderings (must be 1)")
else:
    # Not an error -- it means the measurement notebook has not run yet on any
    # model this account can see.
    trained = [d.name for d in sorted(sess.runs_dir.iterdir())
               if (d / 'summary.json').exists()] if sess.runs_dir.exists() else []
    print('No measured models found.')
    print()
    if trained:
        print(f'{len(trained)} run(s) have finished TRAINING but not MEASUREMENT:')
        for t in trained[:10]:
            print(f'  {t}')
        print()
        print('-> Run NB02 (Phase 0) or NB08 (atlas) to produce the per-sample')
        print('   tables, then re-run this notebook.')
    else:
        print('-> No completed runs at all. Run sess.sync_state(), or finish')
        print('   the training notebooks first.')

## Step 2 — Load the ceilings from NB09

Run NB09 first. Without ceilings, transfer numbers can't be interpreted.

In [ ]:
ceilings = msc.read_json(sess.data_dir / 'analysis' / 'ceilings.json', default=None)
assert ceilings, 'No ceilings found. Run NB09 first.'
print(f'{len(ceilings)} runs have a noise ceiling')

seed1 = msc.representative_runs(runs, require=ceilings)
_missing = sorted(set(msc.ZOO) - set(seed1))
if _missing:
    print(f'[NOTE] {len(_missing)} architecture(s) absent from the transfer '
          f'matrix (no measured seed pair): {", ".join(_missing)}')
    print('       Run NB08 for those runs to include them. See D-15/D-18.')
fam = {m['arch']: m['family'] for m in runs.values()}
print(f'{len(seed1)} architectures available for the transfer matrix')

## Step 3 — The sanity check, on every pair

Scramble one side, re-measure. Agreement must collapse.

Run this **before** looking at the real numbers. Misaligned tables
produce entirely believable results.

**"Collapse" needs a number, and eyeballing one is how you build an
alarm that cries wolf.** A shuffle doesn't leave exactly zero — the
leftover correlation wobbles, with standard deviation `1/√n` ≈ 0.013
here. Across ~78 pairs you should *expect* two or three draws sitting
2–3 wobbles out. That is arithmetic, not a bug.

So a pair is flagged only when the leftover correlation is both
**impossible under shuffling** (|z| > 5) **and large enough to matter**
(|ρ| > 0.10). A real alignment bug doesn't squeak past that pair of
conditions — it lands near ρ ≈ 0.6, roughly 45σ out. The earlier
version of this check used a flat `|T| < 0.05` and failed a healthy
pair at 2.6σ; see **D-17**.

In [ ]:
import itertools
pairs = list(itertools.combinations(sorted(seed1), 2))
print(f'{len(pairs)} architecture pairs -- testing every one\n')

ctrl = []
for a, b in pairs:
    c = msc.analyse_q3_shuffled_control(sess.data_dir, seed1[a], seed1[b],
                                        ceilings, budgets, tau=0.1)
    c.update({'arch_a': a, 'arch_b': b})
    ctrl.append(c)
ctrl = pd.DataFrame(ctrl)
if not len(ctrl):
    print('No architecture pairs available yet. Q3 needs at least two')
    print('architectures measured AND a noise ceiling for each (NB09).')
else:
    # Save BEFORE asserting. If the control ever does fail we want the evidence
    # sitting on HuggingFace to diagnose from, not an exception and an empty
    # analysis folder.
    msc.save_analysis(sess.data_dir, 'q3_shuffled_control', ctrl, sess.hub)

    worst = ctrl.reindex(ctrl.z.abs().sort_values(ascending=False).index)
    print('Ten largest deviations (everything else is smaller):')
    display(worst[['arch_a', 'arch_b', 'spearman_raw', 'z', 'n', 'passed']]
            .head(10).round(4))
    zmax, rfloor = ctrl.z_max.iloc[0], ctrl.rho_floor.iloc[0]
    print(f'\n{ctrl.passed.sum()}/{len(ctrl)} pairs pass')
    print(f'largest |rho| = {ctrl.spearman_raw.abs().max():.4f} '
          f'(|z| = {ctrl.z.abs().max():.1f})')
    print(f'bug threshold = |z| > {zmax:.0f} AND |rho| > {rfloor:.2f}')
    print(f'null SD at this n is ~{ctrl.null_sd.mean():.4f}, so |z| of 2-3 on a '
          f'few of {len(ctrl)} pairs is expected and is NOT a defect.')
    assert ctrl.passed.all(), (
        'Scrambled control FAILED -- shuffling did not destroy the correlation, '
        'so the per-image tables are not paired by sample_idx. Bug, not a finding.')
    print('\nControl passed -- the transfer numbers below are trustworthy.')

## Step 4 — The transfer matrix

`pair_type` is the grouping that makes our prediction testable. H3 says
the ordering should be:

**within-family > across-CNN-family > CNN→Transformer**

because architectures with similar "thinking styles" should agree more.

In [ ]:
TOKEN = {'vit', 'mixer'}
q3 = msc.analyse_q3_transfer(
    sess.data_dir, [(seed1[a], seed1[b]) for a, b in pairs],
    ceilings, budgets, axis='depth', taus=(0.1,), n_boot=1000)

inv_arch = {v: k for k, v in seed1.items()}
q3['arch_a'] = q3.run_a.map(inv_arch); q3['arch_b'] = q3.run_b.map(inv_arch)
q3['fam_a'] = q3.arch_a.map(fam);      q3['fam_b'] = q3.arch_b.map(fam)

def pair_type(r):
    if r.fam_a == r.fam_b:
        return '1_within-family'
    if (r.fam_a in TOKEN) != (r.fam_b in TOKEN):
        return '3_CNN->transformer'
    if r.fam_a in TOKEN and r.fam_b in TOKEN:
        return '4_transformer->transformer'
    return '2_across-CNN-family'
q3['pair_type'] = q3.apply(pair_type, axis=1)

msc.save_analysis(sess.data_dir, 'q3_transfer_matrix', q3, sess.hub)
if len(q3):
    display(q3.groupby('pair_type')[['T', 'spearman_raw', 'jaccard_top10']]
              .agg(['mean', 'std', 'count']).round(3))
else:
    print('No pairs to compare yet.')
print('\nH3 predicts T decreases down this table.')
print('  T ~ 1.0 -> transfer as complete as measurement allows')
print('  T < 0.5 -> architecture-specific; the field assumption is wrong')

## Step 5 — The heatmap (a main figure of the paper)

In [ ]:
archs = sorted(seed1)
M = pd.DataFrame(np.nan, index=archs, columns=archs)
for _, r in q3.iterrows():
    M.loc[r.arch_a, r.arch_b] = r['T']
    M.loc[r.arch_b, r.arch_a] = r['T']
np.fill_diagonal(M.values, 1.0)

fig, ax = plt.subplots(figsize=(10, 8.5))
im = ax.imshow(M.values.astype(float), vmin=0, vmax=1.1, cmap='viridis')
ax.set_xticks(range(len(archs))); ax.set_xticklabels(archs, rotation=90)
ax.set_yticks(range(len(archs))); ax.set_yticklabels(archs)
for i in range(len(archs)):
    for j in range(len(archs)):
        v = M.values[i, j]
        if np.isfinite(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                    color='w' if v < 0.7 else 'k')
ax.set_title('Q3: transfer T(A,B) — 1.0 means "as much agreement as our\n'
             'measurement precision permits"')
plt.colorbar(im, ax=ax, shrink=.8)
plt.tight_layout()
msc.save_figure(fig, sess.data_dir, 'q3_transfer_heatmap', sess.hub)
plt.show()

## Step 6 — Is the predicted ordering there?

A box plot by pair type. If H3 holds, the boxes step downward.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
order = sorted(q3.pair_type.unique())
ax.boxplot([q3[q3.pair_type == p]['T'].dropna() for p in order],
           labels=[p.split('_', 1)[1] for p in order])
ax.axhline(0.8, ls='--', c='g', lw=1, label='H3: within-family > 0.8')
ax.axhline(0.6, ls='--', c='r', lw=1, label='H3: CNN->transformer < 0.6')
ax.set_ylabel('T'); ax.set_title('Q3: does transfer depend on architectural similarity?')
ax.legend(); ax.grid(alpha=.3)
plt.xticks(rotation=15)
plt.tight_layout()
msc.save_figure(fig, sess.data_dir, 'q3_transfer_by_pair_type', sess.hub)
plt.show()

## Step 7 — Does it hold at every τ?

A conclusion that survives only one confidence threshold is not a
conclusion. Slower (bootstraps over the full τ grid).

In [ ]:
def _kind(ab):
    fa, fb = fam[ab[0]], fam[ab[1]]
    if fa == fb:
        return '1_within-family'
    if (fa in TOKEN) != (fb in TOKEN):
        return '3_CNN->transformer'
    if fa in TOKEN and fb in TOKEN:
        return '4_transformer->transformer'
    return '2_across-CNN-family'

# Up to 3 pairs from EACH pair type. The old `pairs[:8]` took the alphabetical
# head, which was eight convnext_femto pairs -- so the "does it hold at every
# tau" check only ever tested one architecture, and the most atypical one at
# that. See D-18.
sample_pairs = msc.stratified_pairs(pairs, _kind, per_kind=3)
print(f'tau curves on {len(sample_pairs)} pairs, stratified by pair type:')
for _k in sorted({_kind(p) for p in sample_pairs}):
    print(f'   {_k}: {sum(1 for p in sample_pairs if _kind(p) == _k)}')
q3t = msc.analyse_q3_transfer(
    sess.data_dir, [(seed1[a], seed1[b]) for a, b in sample_pairs],
    ceilings, budgets, axis='depth', taus=msc.TAU_GRID, n_boot=300)
q3t['pair'] = [f'{inv_arch[a]}->{inv_arch[b]}'
               for a, b in zip(q3t.run_a, q3t.run_b)]
msc.save_analysis(sess.data_dir, 'q3_transfer_tau_curves', q3t, sess.hub)

fig, ax = plt.subplots(figsize=(9, 5))
for p, g in q3t.groupby('pair'):
    ax.plot(g.tau, g['T'], 'o-', label=p)
ax.axhline(0.7, ls='--', c='k', lw=1)
ax.set_xlabel('tau'); ax.set_ylabel('T'); ax.set_title('Q3: stability across tau')
ax.legend(fontsize=7); ax.grid(alpha=.3)
plt.tight_layout()
msc.save_figure(fig, sess.data_dir, 'q3_tau_stability', sess.hub)
plt.show()

## Step 8 — Finish

In [ ]:
# === Push everything and stop ==============================================
# Blocks until HuggingFace confirms. Safe to re-run.
sess.finish()

# D-19: draining the upload queue is NOT the same as the files being on
# HuggingFace, and "[SESSION] done" reads like a confirmation it is not.
# Ask the repository before you close this tab.
#
# D-20: three states, not two. FINISHED and RESUMABLE are both safe -- a run
# paused at epoch 120 whose ckpt_last.pt is on HF loses nothing when you close
# the tab. Only AT RISK (no summary.json AND no checkpoint) needs action.
try:
    _ids = [c['run_id'] for c in (all_cfgs if 'all_cfgs' in dir() else cfgs)]
except NameError:
    _ids = []
if _ids:
    sess.confirm_on_hf(_ids)